In [1]:
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
import joblib
from geopy.distance import geodesic

# Weather impact configurations
WEATHER_IMPACTS = {
    0: {"name": "Clear", "speed_factor": 1.0, "passenger_factor": 1.0, "boarding_time": 1.0},
    1: {"name": "Rain", "speed_factor": 0.7, "passenger_factor": 1.2, "boarding_time": 1.3},
    2: {"name": "Heavy Rain", "speed_factor": 0.5, "passenger_factor": 1.5, "boarding_time": 1.5},
    3: {"name": "Thunderstorm", "speed_factor": 0.4, "passenger_factor": 0.8, "boarding_time": 1.7},
    4: {"name": "Fog", "speed_factor": 0.6, "passenger_factor": 0.9, "boarding_time": 1.4}
}

def load_model_and_data():
    """Load model, scaler and data with error handling"""
    try:
        model = joblib.load('bus_eta_predictor.joblib')
        scaler = joblib.load('standard_scaler.joblib')
        df = pd.read_csv('bus_eta_standard_scaled.csv')
        
        # Fix column name if needed
        if 'passenger_count' not in df.columns and 'passenger_count' in df.columns:
            df.rename(columns={'passenger_count': 'passenger_count'}, inplace=True)
            
        return model, scaler, df
    except Exception as e:
        print(f"Error loading files: {e}")
        exit()

def initialize_stop_data(df):
    """Prepare stop locations and route information"""
    stop_sequence = df.sort_values('stop_sequence')['current_stop_name'].unique()
    stop_locations = {}
    route_distances = {}
    
    for stop in stop_sequence:
        stop_data = df[df['current_stop_name'] == stop].iloc[0]
        next_stop = df[df['current_stop_name'] == stop]['next_stop_name'].values[0]
        
        # Skip if next_stop is not in our sequence
        if next_stop not in stop_sequence:
            print(f"Warning: Next stop '{next_stop}' for '{stop}' not found in stop sequence")
            continue
            
        stop_locations[stop] = {
            'lat': stop_data['current_lat'],
            'lon': stop_data['current_lon'],
            'next_stop': next_stop,
            'typical_passengers': int(df[df['current_stop_name'] == stop]['passenger_count'].median()),
            'weather_tendency': df[df['current_stop_name'] == stop]['weather_condition'].mode()[0]
        }
        
    # Now calculate distances only between valid stops (in kilometers)
    for stop in stop_locations:
        next_stop = stop_locations[stop]['next_stop']
        if next_stop in stop_locations:  # Only if next stop exists
            route_distances[stop] = geodesic(
                (stop_locations[stop]['lat'], stop_locations[stop]['lon']),
                (stop_locations[next_stop]['lat'], stop_locations[next_stop]['lon'])
            ).kilometers  # Changed to kilometers
            
    return stop_sequence, stop_locations, route_distances

def simulate_bus_with_weather():
    """Main simulation function with weather impacts"""
    model, scaler, df = load_model_and_data()
    stop_sequence, stop_locations, route_distances = initialize_stop_data(df)
    
    # Initialize simulation
    current_stop = stop_sequence[0]
    next_stop = stop_locations[current_stop]['next_stop']
    current_speed = 25  # Start at moderate speed (25 km/h)
    passenger_count = stop_locations[current_stop]['typical_passengers']
    weather = stop_locations[current_stop]['weather_tendency']
    distance_remaining = route_distances[current_stop]
    
    # Weather event scheduler
    weather_events = [
        {"time": 15, "weather": 1, "duration": 5},  # Rain after 15 sec
        {"time": 45, "weather": 2, "duration": 10},  # Heavy rain
        {"time": 90, "weather": 3, "duration": 7}    # Thunderstorm
    ]
    current_event = None
    event_start_time = None
    start_time = time.time()
    
    print("🌧️🚌 Starting Bus Simulation with Dynamic Weather (Metric Units)")
    print("-----------------------------------------------------------")
    
    while next_stop in stop_locations:
        # Calculate elapsed time and check weather events
        elapsed = time.time() - start_time
        weather, current_event, event_start_time = update_weather_conditions(
            elapsed, weather_events, current_event, event_start_time, weather, stop_locations[current_stop]['weather_tendency'])
        
        # Get current conditions
        now = datetime.now()
        weather_data = WEATHER_IMPACTS[weather]
        effective_speed = current_speed * weather_data['speed_factor']  # km/h
        
        # Update position and calculate progress
        distance_remaining, progress_ratio, current_lat, current_lon = update_bus_position(
            current_stop, next_stop, stop_locations, distance_remaining, effective_speed)
        
        # Prepare features for prediction
        features = create_feature_vector(
            current_stop, next_stop, now, weather, passenger_count, 
            effective_speed, distance_remaining, current_lat, current_lon, df, model)
        
        # Calculate ETA with proper distance scaling
        eta_minutes = calculate_eta(
            model, scaler, features, distance_remaining, effective_speed, 
            passenger_count, weather_data['boarding_time'])
        
        # Calculate arrival timestamp
        eta_seconds = eta_minutes * 60
        arrival_time = now + timedelta(seconds=eta_seconds)
        
        # Display current status
        display_status(now, weather_data, current_stop, next_stop, distance_remaining, 
                     current_speed, effective_speed, passenger_count, eta_minutes, 
                     progress_ratio, arrival_time)
        
        time.sleep(2)
        
        # Update movement parameters
        current_speed = update_speed(
            current_speed, progress_ratio, weather_data['speed_factor'], distance_remaining)
        
        # Handle arrivals
        if distance_remaining <= 0.01:  # 0.01 km = 10 meters
            current_stop, next_stop, passenger_count, current_speed, distance_remaining = handle_arrival(
                current_stop, next_stop, stop_locations, route_distances, passenger_count, 
                weather_data['passenger_factor'], weather_data['boarding_time'])

def update_weather_conditions(elapsed, weather_events, current_event, event_start_time, current_weather, default_weather):
    """Manage weather event triggering and clearing"""
    if current_event is None:
        for event in weather_events:
            if elapsed >= event['time'] and not event.get('triggered', False):
                current_event = event
                event_start_time = time.time()
                current_weather = event['weather']
                event['triggered'] = True
                print(f"\n⚠️ WEATHER ALERT: {WEATHER_IMPACTS[current_weather]['name']} detected!")
                break
    elif time.time() - event_start_time > current_event['duration']:
        print(f"\n🌤️ Weather cleared: Back to normal conditions")
        current_weather = default_weather
        current_event = None
        
    return current_weather, current_event, event_start_time

def update_bus_position(current_stop, next_stop, stop_locations, distance_remaining, effective_speed):
    """Calculate new bus position and progress (in km)"""
    # Update distance first (convert speed from km/h to km per 2 seconds)
    speed_km_per_2sec = effective_speed / 1800  # 3600 seconds/hour ÷ 2
    new_distance = max(0, distance_remaining - speed_km_per_2sec)
    
    # Calculate progress ratio
    total_distance = geodesic(
        (stop_locations[current_stop]['lat'], stop_locations[current_stop]['lon']),
        (stop_locations[next_stop]['lat'], stop_locations[next_stop]['lon'])
    ).kilometers
    progress_ratio = 1 - (new_distance / total_distance)
    
    # Calculate interpolated position
    current_lat = stop_locations[current_stop]['lat'] + (
        stop_locations[next_stop]['lat'] - stop_locations[current_stop]['lat']) * progress_ratio
    current_lon = stop_locations[current_stop]['lon'] + (
        stop_locations[next_stop]['lon'] - stop_locations[current_stop]['lon']) * progress_ratio
        
    return new_distance, progress_ratio, current_lat, current_lon

def create_feature_vector(current_stop, next_stop, now, weather, passenger_count, 
                         speed, distance, lat, lon, df, model):
    """Prepare the feature vector for prediction"""
    features = pd.DataFrame([{
        'current_stop_name': current_stop,
        'next_stop_name': next_stop,
        'day_of_week': now.weekday(),
        'is_holiday': 0,
        'is_peak_hour': int(7 <= now.hour <= 9 or 16 <= now.hour <= 19),
        'weather_condition': weather,
        'passenger_count': passenger_count,
        'current_speed': speed,
        'distance_to_next_stop': distance,
        'current_lat': lat,
        'current_lon': lon
    }])
    
    # Ensure all model features are present
    for col in model.feature_names_in_:
        if col not in features.columns:
            if col in ['current_stop_name', 'next_stop_name']:
                features[col] = df[col].mode()[0]
            else:
                features[col] = df[col].median()
    
    return features[model.feature_names_in_]

def calculate_eta(model, scaler, features, distance_remaining, effective_speed, passenger_count, boarding_factor):
    """Calculate ETA with proper distance scaling (returns minutes)"""
    try:
        eta_scaled = model.predict(features)[0]
        base_eta = float(scaler.inverse_transform([[eta_scaled]])[0][0])
        
        # Physics-based minimum ETA (convert km and km/h to minutes)
        min_eta = (distance_remaining / max(0.1, effective_speed)) * 60
        
        if distance_remaining < 0.1:  # Final approach (100 meters)
            # Combine movement time with boarding time estimate
            eta = max(0.1, min_eta + (passenger_count * 0.01 * boarding_factor))
        else:
            # Normal conditions - blend model and physics
            eta = max(0.1, min(base_eta, min_eta * 2))
            
    except Exception as e:
        # Fallback to physics calculation
        eta = (distance_remaining / max(0.1, effective_speed)) * 60
        eta = max(0.1, eta)
    
    return round(eta, 1)

def update_speed(current_speed, progress_ratio, speed_factor, distance_remaining):
    """Calculate new speed with realistic acceleration/deceleration (in km/h)"""
    if distance_remaining < 0.1:  # Approaching stop (100 meters)
        acceleration = -3.0 * speed_factor  # Strong deceleration (km/h per 2 sec)
    elif progress_ratio < 0.3:    # Starting from stop
        acceleration = 2.5 * speed_factor
    elif progress_ratio > 0.8:    # Preparing to stop
        acceleration = -1.2 * speed_factor
    else:                         # Cruising
        acceleration = 0.3 if current_speed < 40 else 0
        
    new_speed = current_speed + acceleration
    return max(10, min(60, new_speed))  # Keep between 10-60 km/h

def handle_arrival(current_stop, next_stop, stop_locations, route_distances, passenger_count, passenger_factor, boarding_time):
    """Process bus arrival at a stop"""
    print(f"\n🚏 ARRIVED AT {next_stop} (Boarding delay: {boarding_time:.1f}x normal)")
    time.sleep(boarding_time * 1)  # Simulate boarding
    
    # Passenger changes
    passengers_leaving = min(passenger_count, np.random.poisson(3))
    passengers_boarding = np.random.poisson(4 * passenger_factor)
    passenger_count = max(0, passenger_count - passengers_leaving + passengers_boarding)
    
    # Initialize default values
    new_current = next_stop
    new_next = stop_locations[new_current].get('next_stop')
    new_speed = 25  # Default reset speed (25 km/h)
    new_distance = 0  # Default distance
    
    if new_next:
        new_distance = route_distances[new_current]
    else:
        print("🏁 End of route reached!")
        
    return new_current, new_next, passenger_count, new_speed, new_distance

def display_status(now, weather_data, current_stop, next_stop, distance, 
                  base_speed, effective_speed, passengers, eta, progress, arrival_time):
    """Display the current simulation status"""
    print(f"\n⏰ Current Time: {now.strftime('%H:%M:%S')} | 🌧️ {weather_data['name']}")
    print(f"📍 Route: {current_stop} → {next_stop}")
    print(f"📏 Distance: {distance:.3f} km | 🚦 Speed: {effective_speed:.1f} km/h (Base: {base_speed:.1f})")
    print(f"👥 Passengers: {passengers} | Boarding: {weather_data['boarding_time']:.1f}x normal")
    print(f"⏳ Predicted ETA: {eta:.1f} min (Weather impact: {1/weather_data['speed_factor']:.1f}x normal)")
    print(f"🕒 Expected Arrival: {arrival_time.strftime('%H:%M:%S')}")
    
    # Visual progress bar
    bus_pos = min(int(20 * progress), 19)
    print(f"[{'·'*bus_pos}🚌{'·'*(19-bus_pos)}] {int(progress*100)}%")

# Run the simulation
if __name__ == "__main__":
    simulate_bus_with_weather()

🌧️🚌 Starting Bus Simulation with Dynamic Weather (Metric Units)
-----------------------------------------------------------
[LightGBM] [Warning] min_data_in_leaf is set=94, min_child_samples=81 will be ignored. Current value: min_data_in_leaf=94
[LightGBM] [Warning] feature_fraction is set=0.9722793276963305, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9722793276963305
[LightGBM] [Warning] lambda_l1 is set=4.905310569171177e-07, reg_alpha=0.0 will be ignored. Current value: lambda_l1=4.905310569171177e-07
[LightGBM] [Warning] lambda_l2 is set=8.094332988586253e-06, reg_lambda=0.0 will be ignored. Current value: lambda_l2=8.094332988586253e-06
[LightGBM] [Warning] bagging_fraction is set=0.7481514265968381, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7481514265968381
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3

⏰ Current Time: 01:00:40 | 🌧️ Clear
📍 Route: 4 → 1
📏 Distance: 1.

: 